# Tugas 3 — Visualisasi Transaksi POS Tahun 2025

Empat grafik disatukan dalam satu kanvas Matplotlib berbentuk persegi (1:1).

## Persiapan Data

Dataset dibaca dari folder `data/`. Data duplikat dihapus, format harga dibersihkan, dan tanggal diubah menjadi tipe waktu.

Penjualan bulanan serta produk terlaris memakai transaksi `SELESAI` dengan jumlah 1–12 unit. Penjualan bersih dihitung setelah diskon dan sebelum PPN.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.ticker import FuncFormatter

ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "data").exists() and (ROOT_DIR.parent / "data").exists():
    ROOT_DIR = ROOT_DIR.parent

DATA_PATH = ROOT_DIR / "data" / "paket_a_transaksi_pos_2024_2025.csv"
data = pd.read_csv(DATA_PATH, dtype=str)
data = data.drop_duplicates().copy()

def bersihkan_harga(nilai):
    nilai = str(nilai).strip()
    if "," in nilai:
        nilai = nilai.replace(".", "").replace(",", ".")
    else:
        nilai = nilai.replace(".", "")
    return float(nilai)

data["harga_satuan"] = data["harga_satuan"].map(bersihkan_harga)
data["jumlah"] = data["jumlah"].astype(int)
data["diskon_persen"] = data["diskon_persen"].astype(int)
data["tanggal"] = pd.to_datetime(
    data["tanggal"], format="mixed", dayfirst=True
)

data_2025 = data.loc[data["tanggal"].dt.year == 2025].copy()
data_jual = data_2025.loc[
    (data_2025["status"] == "SELESAI")
    & (data_2025["jumlah"].between(1, 12))
].copy()

data_jual["Penjualan_Bersih"] = (
    data_jual["harga_satuan"]
    * data_jual["jumlah"]
    * (1 - data_jual["diskon_persen"] / 100)
)

print("Jumlah data setelah menghapus duplikat:", len(data))
print("Jumlah transaksi tahun 2025:", len(data_2025))
print("Transaksi SELESAI dengan jumlah 1–12 unit:", len(data_jual))
print("\nDistribusi status transaksi tahun 2025:")
print(data_2025["status"].value_counts())

## Visualisasi Gabungan 2×2

Gambar memuat penjualan bersih per bulan, sepuluh produk terlaris, proporsi status transaksi, serta hubungan jumlah pembelian dengan diskon.

Status dihitung dari seluruh transaksi tahun 2025. Ukuran titik pada grafik keempat menunjukkan frekuensi transaksi pada setiap kombinasi jumlah dan diskon.

In [ ]:
# Agregasi untuk empat grafik
nama_bulan = ["Jan", "Feb", "Mar", "Apr", "Mei", "Jun",
              "Jul", "Agu", "Sep", "Okt", "Nov", "Des"]

penjualan_bulanan = (
    data_jual.groupby(data_jual["tanggal"].dt.month)["Penjualan_Bersih"]
    .sum()
    .reindex(range(1, 13), fill_value=0)
)

produk_terlaris = (
    data_jual.groupby("kode_produk")["jumlah"]
    .sum()
    .nlargest(10)
    .sort_values()
)

jumlah_status = data_2025["status"].value_counts()
status_selesai = int(jumlah_status.get("SELESAI", 0))
status_batal = int(jumlah_status.get("BATAL", 0))
status_retur = int(jumlah_status.get("RETUR", 0))
tidak_selesai = status_batal + status_retur
jumlah_total = status_selesai + tidak_selesai

hubungan = (
    data_jual.groupby(["jumlah", "diskon_persen"])
    .size()
    .reset_index(name="frekuensi")
)

# Satu kanvas persegi dengan empat subplot
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 15,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "figure.facecolor": "white",
    "axes.facecolor": "#FBFCFE",
    "axes.edgecolor": "#D9E0E8",
    "axes.linewidth": 0.8,
    "grid.color": "#DCE3EB",
    "grid.linewidth": 0.8,
})

fig, axes = plt.subplots(2, 2, figsize=(14, 14))
fig.patch.set_facecolor("white")
fig.suptitle(
    "VISUALISASI TRANSAKSI POS · 2025",
    fontsize=23, fontweight="bold", y=0.965, color="#172536"
)
fig.text(
    0.5, 0.935,
    "Ringkasan penjualan, produk, status transaksi, dan pola diskon",
    ha="center", fontsize=11, color="#657487"
)

# 1. Penjualan bersih per bulan
ax = axes[0, 0]
nilai_juta = penjualan_bulanan.values / 1_000_000
bars = ax.bar(nama_bulan, nilai_juta, color="#3569A8", width=0.68, zorder=3)
ax.set_title("01  Penjualan Bersih per Bulan", loc="left", pad=14)
ax.set_ylabel("Penjualan bersih (juta rupiah)")
ax.set_ylim(0, max(nilai_juta) * 1.22)
ax.grid(axis="y", zorder=0)
ax.set_axisbelow(True)
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:.0f}"))
for bar, value in zip(bars, nilai_juta):
    ax.annotate(
        f"{value:.1f}",
        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 5), textcoords="offset points",
        ha="center", va="bottom", fontsize=8, color="#34465A"
    )
ax.spines[["top", "right"]].set_visible(False)

# 2. Sepuluh produk terlaris
ax = axes[0, 1]
bars = ax.barh(
    produk_terlaris.index,
    produk_terlaris.values,
    color="#4D8C78", height=0.68, zorder=3
)
ax.set_title("02  10 Produk Terlaris", loc="left", pad=14)
ax.set_xlabel("Total unit terjual")
ax.set_xlim(0, produk_terlaris.max() * 1.18)
ax.grid(axis="x", zorder=0)
ax.set_axisbelow(True)
for bar, value in zip(bars, produk_terlaris.values):
    ax.annotate(
        f"{value}",
        (bar.get_width(), bar.get_y() + bar.get_height() / 2),
        xytext=(6, 0), textcoords="offset points",
        ha="left", va="center", fontsize=9, color="#34465A"
    )
ax.spines[["top", "right"]].set_visible(False)

# 3. Proporsi status transaksi
ax = axes[1, 0]
wedges, _, autotexts = ax.pie(
    [status_selesai, tidak_selesai],
    labels=None,
    autopct="%1.1f%%",
    startangle=90,
    counterclock=False,
    colors=["#4D8C78", "#D49A48"],
    wedgeprops={"width": 0.42, "edgecolor": "white", "linewidth": 2},
    pctdistance=0.78,
    textprops={"fontsize": 10, "color": "#34465A"},
    radius=0.60,
    center=(-0.45, 0),
)
for text in autotexts:
    text.set_fontweight("bold")
    text.set_color("#172536")
    text.set_fontsize(10)
ax.text(
    -0.45, 0.07, f"{jumlah_total:,}".replace(",", "."),
    ha="center", va="center", fontsize=17, fontweight="bold",
    color="#172536"
)
ax.set_title("03  Proporsi Status Transaksi", loc="left", pad=14)
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-1.25, 1.45)
ax.set_ylim(-1.00, 1.00)
ax.text(
    0.62, 0.69, "SELESAI", transform=ax.transAxes,
    ha="left", va="center", fontsize=10, fontweight="bold", color="#4D8C78"
)
ax.text(
    0.62, 0.59,
    f"{status_selesai:,} transaksi\n{status_selesai / jumlah_total:.1%}",
    transform=ax.transAxes, ha="left", va="center",
    fontsize=9, color="#526174"
)
ax.text(
    0.62, 0.36, "BATAL", transform=ax.transAxes,
    ha="left", va="center", fontsize=10, fontweight="bold", color="#D49A48"
)
ax.text(
    0.62, 0.27, f"{status_batal} transaksi", transform=ax.transAxes,
    ha="left", va="center", fontsize=9, color="#526174"
)
ax.text(
    0.62, 0.11, "RETUR", transform=ax.transAxes,
    ha="left", va="center", fontsize=10, fontweight="bold", color="#D49A48"
)
ax.text(
    0.62, 0.02, f"{status_retur} transaksi", transform=ax.transAxes,
    ha="left", va="center", fontsize=9, color="#526174"
)

# 4. Hubungan jumlah pembelian dengan diskon
ax = axes[1, 1]
ax.scatter(
    hubungan["jumlah"],
    hubungan["diskon_persen"],
    s=hubungan["frekuensi"] * 1.05 + 22,
    color="#8A5AA6",
    alpha=0.62,
    edgecolors="white",
    linewidths=1.0,
    zorder=3,
)
ax.set_title("04  Jumlah Pembelian dan Diskon", loc="left", pad=14)
ax.set_xlabel("Jumlah barang dibeli (unit)")
ax.set_ylabel("Diskon (%)")
ax.set_xticks(sorted(hubungan["jumlah"].unique()))
ax.set_yticks(sorted(hubungan["diskon_persen"].unique()))
ax.set_xlim(0, 13)
ax.set_ylim(-2, 22)
ax.grid(zorder=0)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.text(
    0.02, 0.02, "Ukuran titik menunjukkan frekuensi transaksi",
    transform=ax.transAxes, fontsize=8.5, color="#657487", va="bottom"
)

fig.text(
    0.5, 0.035,
    "Penjualan dan produk: status SELESAI, jumlah 1–12 unit, "
    "nilai setelah diskon dan sebelum PPN.  |  Duplikat dihapus.",
    ha="center", va="center", fontsize=8.0, color="#657487"
)
fig.subplots_adjust(
    left=0.09, right=0.95, top=0.89, bottom=0.09,
    wspace=0.30, hspace=0.34
)

OUTPUT_DIR = ROOT_DIR / "outputs" / "visualisasi"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
png_path = OUTPUT_DIR / "visualisasi_gabungan_2025.png"
svg_path = OUTPUT_DIR / "visualisasi_gabungan_2025.svg"

# Tanpa pemotongan bounding box: ukuran gambar tetap 1:1
fig.savefig(png_path, dpi=220, facecolor="white")
fig.savefig(svg_path, format="svg", facecolor="white")
print("Gambar disimpan ke:", png_path)
print("Versi SVG disimpan ke:", svg_path)
plt.show()

## Hasil

Satu gambar PNG persegi disimpan di `outputs/visualisasi/visualisasi_gabungan_2025.png`. Versi SVG juga dibuat agar grafik tetap tajam ketika diperbesar.